<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 5 — Python Toolkit</div>
</div>

#### Spectra with rfft, spectrogram and welch, uneven times to a regular grid, shifting signals, and timing code.
***

## Why this notebook

Five pieces of Python for today's lab and for Exercise 2. Fifteen minutes; skim what you know. Section 6 has optional practice with numbers to check against.

1. `np.fft.rfft` and `rfftfreq`: a spectrum in two lines;
2. `scipy.signal.spectrogram` and `welch`: spectra in windows (the short-time Fourier transform), and their average;
3. from uneven times to a regular grid: `np.interp` and `CubicSpline`;
4. `np.roll`: shifting a signal in time (surrogates);
5. timing your code, and why loops over samples are slow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import CubicSpline
import time

## <span style="color:#B5121B">1. A spectrum in two lines</span>

`rfft` returns one complex number per frequency; `rfftfreq(n, d)` gives those frequencies for `n` samples spaced `d` seconds apart. The power at each frequency is $|X|^2$.

In [ ]:
fs = 10.0                                  # samples per second
t = np.arange(0, 20, 1 / fs)               # 20 s
x = np.sin(2 * np.pi * 1.5 * t)            # a 1.5 Hz sine
X = np.fft.rfft(x)
f = np.fft.rfftfreq(len(x), 1 / fs)
print(len(x), "samples ->", len(f), "frequencies from", f[0], "to", f[-1], "Hz, spacing", f[1], "Hz")
print("peak at", f[np.argmax(np.abs(X)**2)], "Hz")

In [ ]:
# Your turn: add a 3 Hz sine of half the amplitude to x and find both peaks

## <span style="color:#B5121B">2. Spectra in windows, and their average</span>

`signal.spectrogram(x, fs=..., window="hann", nperseg=n, noverlap=n // 2)` is the short-time Fourier transform: it cuts `x` into windows of `n` samples (half-overlapping), and returns the frequencies `f`, the window centres `tt`, and `S`, with one column per window: `S[:, j]` is the spectrum of window `j`.

`signal.welch(x, fs=..., nperseg=n)` does the same and **averages the columns over time**: an averaged spectrum, much less noisy than the spectrum of the whole record. Both return a power spectral density (units² per Hz).

In [ ]:
rng = np.random.default_rng(0)
noisy = x + rng.standard_normal(len(x))
f_s, tt, S = signal.spectrogram(noisy, fs=fs, window="hann", nperseg=64, noverlap=32)
print(S.shape, "= (frequencies, windows)")
f_w, P_w = signal.welch(noisy, fs=fs, nperseg=64)
print("welch is the time-average of the spectrogram:", np.allclose(S.mean(axis=1), P_w))
plt.semilogy(f_w, P_w); plt.xlabel("Frequency (Hz)"); plt.ylabel("PSD"); plt.show()

## <span style="color:#B5121B">3. From uneven times to a regular grid</span>

Values known at uneven times (events) can be put on a regular grid by interpolation. `np.interp` is linear; `CubicSpline` is smooth (Lecture 4).

In [ ]:
t_ev = np.array([0.0, 0.9, 2.1, 2.9, 4.2, 5.0])     # uneven times
v_ev = np.array([1.0, 1.2, 1.1, 0.9, 1.0, 1.3])     # a value at each
grid = np.arange(0, 5.0, 0.25)
print(np.round(np.interp(grid, t_ev, v_ev), 3))
print(np.round(CubicSpline(t_ev, v_ev)(grid), 3))

## <span style="color:#B5121B">4. Shifting a signal in time</span>

`np.roll(x, k)` shifts `x` by `k` samples and wraps the end round to the start. Everything about the signal stays the same except *when* things happen, which is what a surrogate test needs.

In [ ]:
a = np.arange(8)
print(np.roll(a, 3))                       # [5 6 7 0 1 2 3 4]
print(np.allclose(np.roll(np.roll(a, 3), -3), a))   # rolling back undoes it

## <span style="color:#B5121B">5. Timing code</span>

`time.perf_counter()` before and after. A Python loop over every sample is often 100 times slower than the same operation on whole arrays.

In [ ]:
big = rng.standard_normal(1_000_000)
t0 = time.perf_counter(); s = 0.0
for v in big: s += v * v
t1 = time.perf_counter(); s2 = np.sum(big * big); t2 = time.perf_counter()
print(f"loop {t1 - t0:.3f} s; numpy {t2 - t1:.4f} s; same answer: {np.isclose(s, s2)}")

## <span style="color:#FD9029">6. Optional practice</span>

Four short exercises, each with a **Check**.

**P1.** A 0.25 Hz sine sampled 4 times a second for 2 minutes. What is the frequency spacing of `rfftfreq`, and at what frequency is the peak?

In [ ]:
# P1

**Check:** spacing 0.00833 Hz (1/120 s); peak at 0.25 Hz.

**P2.** With `noisy` from Section 2, compare `nperseg=32` and `nperseg=128`. Which has the narrower peak at 1.5 Hz? What is the frequency spacing of each?

In [ ]:
# P2

**Check:** spacings 0.3125 and 0.078 Hz; peaks at 1.56 and 1.48 Hz (the nearest grid points to 1.5); 128 gives the narrower peak.

**P3.** Beat times `beats = np.cumsum(np.full(20, 0.8))` (a steady 0.8 s rhythm). Compute the intervals, put them on a 4 Hz grid with `np.interp`, and check every interpolated value is 0.8.

In [ ]:
# P3

**Check:** 58 grid points, all equal to 0.8 (True).

**P4.** Write a function `best_lag(x, y, maxlag)` that returns the lag (in samples, from −maxlag to +maxlag) at which `np.roll(y, -lag)` correlates best with `x`. Test it with `y = np.roll(x, 7)`.

In [ ]:
# P4

**Check:** 7. (A sine repeats, so lags a whole period apart also fit: a reason to keep `maxlag` shorter than one period.)

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>